In [56]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [57]:
data2= r"C:\Users\rajeshkumar.t\Desktop\ML\Product_Order_id.csv"
df= pd.read_csv(data2)
print(df.columns)

Index(['order_external_id', 'status', 'seller_id',
       'promotion_discount_adjustment', 'price_change_adjustment',
       'unit_creation_date_time', 'order_id', 'account_id',
       'bank_offer_adjustment', 'courier_name', 'deliver_date_time',
       'new_customer_flag', 'cancel_date_time', 'return_completed_date_time',
       'listing_price', 'product_title', 'promise_breach',
       'analytic_business_unit', 'analytic_category', 'analytic_vertical',
       'city_tier', 'cms_vertical'],
      dtype='object')


In [58]:
df_clean = df[~df['status'].isin(['CANCELLED', 'RETURNED'])].copy()
df_clean['unit_creation_date_time'] = pd.to_datetime(df_clean['unit_creation_date_time'], dayfirst=True)
max_date = df_clean['unit_creation_date_time'].max()
split_date = max_date - pd.Timedelta(days=90)
obs_df = df_clean[df_clean['unit_creation_date_time'] < split_date]
features = obs_df.groupby('account_id').agg(
    frequency= ('order_external_id' , 'nunique'),
    avg_spend =('listing_price', 'mean'),
    city_tier = ('city_tier', lambda x: x.mode()[0] if not x.mode().empty else 1),
    promo_ratio= ('promotion_discount_adjustment', lambda x: (x>0).mean()),
    recency_days= ('unit_creation_date_time',lambda x: (split_date - x.max()).days),
    customer_age=('unit_creation_date_time', lambda x:(split_date - x.min()).days)
)
future_df = df_clean[df_clean['unit_creation_date_time'] > split_date]
target = future_df.groupby('account_id')['listing_price'].sum().rename('future_clv')
data = features.join(target).fillna(0)

X= data.drop('future_clv', axis=1)
y= data['future_clv']

X = pd.get_dummies(X, columns=['city_tier'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state = 42)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

data['predicted_CLV_90d'] = model.predict(X)
print(data[['predicted_CLV_90d']].sort_values(by='predicted_CLV_90d', ascending=False).head(10))

    
                        


                                      predicted_CLV_90d
account_id                                             
ACC2485E745085849FCBB4A50F28EC82088              206.05
ACCB481F7CD5A574BB19D4256717BDD5595               72.91
ACC052D1BC97BF94F42A563521BE0F44AA4O               0.00
ACC0884E7D9F85946D0B845B0B8AA1D45FD                0.00
ACC0627AB4320124B94A9130507A0673EBF2               0.00
ACC0A8EF7F7FB5E4DBA9144605325488137                0.00
ACC16ACDF2E959F4368AFF170245292A013X               0.00
ACC208107F60D5241E1A345645E4F3E7795                0.00
ACC0684A7B653574DDBA345999A422D4B61                0.00
ACC242D1997F3724EB3B22C5EE2667752E1A               0.00


In [59]:
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importances': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importances', ascending=False)

In [60]:
comparison = data.groupby('city_tier')['future_clv'].agg(['mean', 'count', 'sum'])
comparison.columns = ['Avg_Future_Spend', 'Customer_Count', 'Total_Potential_Revenue']
print(comparison.sort_values(by='Avg_Future_Spend',ascending=False))

                 Avg_Future_Spend  Customer_Count  Total_Potential_Revenue
city_tier                                                                 
Tier 2                      158.5               2                    317.0
1                             0.0               2                      0.0
Metro                         0.0              12                      0.0
Tier 1A                       0.0               5                      0.0
Tier 1B                       0.0               4                      0.0
Tier 3 & Others               0.0              40                      0.0
